In [ ]:
import numpy as np
import torch
import time
from scipy import sparse as sp
from scipy.linalg import eigh
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

pauli_matrices = {
    1: np.array([[0, 1], [1, 0]], dtype=complex),
    2: np.array([[0, -1j], [1j, 0]], dtype=complex),
    3: np.array([[1, 0], [0, -1]], dtype=complex),
    0: np.eye(2, dtype=complex)
}

pauli_matrices_sparse = {
    key: sp.csr_matrix(value) for key, value in pauli_matrices.items()
}

def pauli_site(x, i, L):
    if L == 1:
        return pauli_matrices_sparse[x]
    if i == 1:
        return sp.kron(pauli_matrices_sparse[x], sp.eye(2**(L-1), format='csr'))
    else:
        return sp.kron(sp.eye(2), pauli_site(x, i-1, L-1), format='csr')

def pauli_int(spin, site, L):
    if site == L:  
        op = sp.eye(2**L, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == L or i == 1:
                op = op @ pauli_site(spin, i, L)
        return op
    else:
        op = sp.eye(2**L, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == site or i == site+1:
                op = op @ pauli_site(spin, i, L)
        return op

def xxz_hamiltonian(L, delta=1):
    dim = 2**L  
    H = sp.csr_matrix((dim, dim), dtype=complex)

    for site in range(1, L+1):
        H += pauli_int(1, site, L)
        H += pauli_int(2, site, L)
        H += delta * pauli_int(3, site, L)
    
    return H
    
def next_nearest_neighbor_interaction(L):
    dim = 2**L
    H1 = sp.csr_matrix((dim, dim), dtype=complex)
    
    for n in range(1, L+1):
        n2 = n + 2
        if n2 > L:
            n2 = n2 - L

        H1 += pauli_site(1, n, L) @ pauli_site(1, n2, L)
        H1 += pauli_site(2, n, L) @ pauli_site(2, n2, L)
        H1 += pauli_site(3, n, L) @ pauli_site(3, n2, L)
    
    return H1

def build_full_hamiltonian(L, delta=1, lambda_val=0.3):
    H0 = xxz_hamiltonian(L, delta)
    H1 = next_nearest_neighbor_interaction(L)
    H = H0 + lambda_val * H1
    return H

def local_ops(dtype=torch.complex64, device="cpu"):
    Sz = torch.tensor([[0.5, 0.0],
                       [0.0, -0.5]], dtype=dtype, device=device)
    Sp = torch.tensor([[0.0, 1.0],
                       [0.0, 0.0]], dtype=dtype, device=device)
    Sm = torch.tensor([[0.0, 0.0],
                       [1.0, 0.0]], dtype=dtype, device=device)
    Sx = torch.tensor([[0.0, 1.0],
                       [1.0, 0.0]], dtype=dtype, device=device)
    Sy = torch.tensor([[0.0, -1j],
                       [1j, 0.0]], dtype=dtype, device=device)
    I2 = torch.eye(2, dtype=dtype, device=device)
    return I2, Sx, Sy, Sz, Sp, Sm

def precompute_perms(L: int):
    perms = []
    inv_perms = []
    axes = list(range(L))
    for s in range(L):
        p = [s] + [a for a in axes if a != s]
        inv = np.argsort(p)
        perms.append(p)
        inv_perms.append(inv.tolist())
    return perms, inv_perms

def apply_one_site(op2, site, vec, L, perms, inv_perms):
    psi = vec.reshape([2]*L).permute(perms[site]).reshape(2, -1)
    psi2 = op2 @ psi
    out = psi2.reshape([2]*L).permute(inv_perms[site]).reshape(-1)
    return out

def B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms):
    dtype, device = psi.dtype, psi.device
    ii = torch.tensor(1j, dtype=dtype, device=device)

    V11 = psi.clone()
    V12 = torch.zeros_like(psi)
    V21 = torch.zeros_like(psi)
    V22 = psi.clone()
    
    for site in range(L):
        SzV11 = apply_one_site(Sz, site, V11, L, perms, inv_perms)
        SzV12 = apply_one_site(Sz, site, V12, L, perms, inv_perms)
        SzV21 = apply_one_site(Sz, site, V21, L, perms, inv_perms)
        SzV22 = apply_one_site(Sz, site, V22, L, perms, inv_perms)

        SmV21 = apply_one_site(Sm, site, V21, L, perms, inv_perms)
        SmV22 = apply_one_site(Sm, site, V22, L, perms, inv_perms)
        SpV11 = apply_one_site(Sp, site, V11, L, perms, inv_perms)
        SpV12 = apply_one_site(Sp, site, V12, L, perms, inv_perms)

        L11V11 = u * V11 + ii * SzV11
        L11V12 = u * V12 + ii * SzV12
        L22V21 = u * V21 - ii * SzV21
        L22V22 = u * V22 - ii * SzV22
        
        L12V21 = ii * SmV21
        L12V22 = ii * SmV22
        L21V11 = ii * SpV11
        L21V12 = ii * SpV12

        N11 = L11V11 + L12V21
        N12 = L11V12 + L12V22
        N21 = L21V11 + L22V21
        N22 = L21V12 + L22V22
        
        V11, V12, V21, V22 = N11, N12, N21, N22
    
    return V12

def vacuum_state(L, dtype=torch.complex64, device="cpu"):
    dim = 2**L
    v = torch.zeros(dim, dtype=dtype, device=device)
    v[0] = 1.0 + 0.0j
    return v

def bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True):
    psi = vacuum_state(L, dtype=I2.dtype, device=I2.device)
    
    for layer_params in params_list:
        u = layer_params[0] + 1j * layer_params[1]
        psi = B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms)
    
    if normalize:
        norm = torch.sqrt(torch.real(torch.vdot(psi, psi)))
        if norm > 1e-15:
            psi = psi / norm
    
    return psi

def exact_diagonalization(H, num_states=5):  
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
        
    eigenvalues, eigenvectors = eigh(H_dense)
    
    results = []
    for i in range(min(num_states, len(eigenvalues))):
        results.append({
            'energy': eigenvalues[i],
            'state': eigenvectors[:, i]
        })
    
    return results

def pytorch_loss_func(params, H_tensor, ground_state_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, 
                     M_number=3, beta_penalty=100.0):

    params_per_layer = 2
    total_layers = len(params) // params_per_layer
    params_list = []
    
    for layer in range(total_layers):
        start_idx = layer * params_per_layer
        re_part = params[start_idx]
        im_part = params[start_idx + 1]
        params_list.append([re_part, im_part])

    psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)

    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    energy_expectation = energy / norm2

    ground_overlap = torch.abs(torch.vdot(psi, ground_state_tensor))
    normalized_overlap_sq = (ground_overlap ** 2) / norm2

    total_loss = energy_expectation + beta_penalty * normalized_overlap_sq
    
    return total_loss

def torch_wrapper_loss(params_numpy, H_tensor, ground_state_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, 
                      M_number, beta_penalty):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)

    loss = pytorch_loss_func(params, H_tensor, ground_state_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, 
                           M_number, beta_penalty)

    loss.backward()

    loss_value = loss.item()
    grad_value = params.grad.numpy().astype(np.float64)
    
    return loss_value, grad_value

def optimize_excited_state_fast(L=6, delta=1.0, lambda_val=0.3, M_number=3, 
                               num_attempts=5, maxiter=6000, device="cpu"):

    print(f"{L} sites")
    print(f"Anisotropy parameter Δ={delta}")
    print(f"Next-nearest neighbor interaction λ={lambda_val}")
    print(f"Number of Bethe layers (M_number): {M_number}")
    print(f"Number of random initialization attempts: {num_attempts}")
    print(f"Device: {device}")

    H = build_full_hamiltonian(L, delta=delta, lambda_val=lambda_val)

    exact_states = exact_diagonalization(H, num_states=4)
    ground_energy = exact_states[0]['energy']
    excited1_energy = exact_states[1]['energy']
    excited2_energy = exact_states[2]['energy']
    excited3_energy = exact_states[3]['energy']
    
    print(f"Exact ground state energy: {ground_energy:.8f}")
    print(f"Exact first excited state energy: {excited1_energy:.8f}")
    print(f"Exact second excited state energy: {excited2_energy:.8f}")
    print(f"Exact third excited state energy: {excited3_energy:.8f}")

    I2, Sx, Sy, Sz, Sp, Sm = local_ops(dtype=torch.complex128, device=device)
    perms, inv_perms = precompute_perms(L)

    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    H_tensor = torch.tensor(H_dense, dtype=torch.complex128, device=device)
    ground_state_tensor = torch.tensor(exact_states[0]['state'], 
                                       dtype=torch.complex128, device=device)
    excited1_state_tensor = torch.tensor(exact_states[1]['state'], 
                                         dtype=torch.complex128, device=device)
    excited2_state_tensor = torch.tensor(exact_states[2]['state'], 
                                         dtype=torch.complex128, device=device)
    excited3_state_tensor = torch.tensor(exact_states[3]['state'], 
                                         dtype=torch.complex128, device=device)

    best_energy = float('inf')
    best_params = None
    best_psi = None
    best_fidelity = 0.0  

    beta_penalty_values = [5, 10.0, 50.0]
    
    for beta_penalty in beta_penalty_values:
        print(f"\nOptimizing with penalty coefficient beta_penalty = {beta_penalty:.1f}:")
        
        for attempt in range(num_attempts):
            print(f"  Attempt {attempt+1}/{num_attempts}:")

            params_per_layer = 2
            total_params = M_number * params_per_layer
            x0 = np.zeros(total_params, dtype=np.float64)
            
            for layer in range(M_number):
                start_idx = layer * params_per_layer
                x0[start_idx:start_idx+2] = np.random.normal(-1, 1, 2)

            def loss_func_wrapper(p):
                return torch_wrapper_loss(p, H_tensor, ground_state_tensor, L, 
                                        I2, Sz, Sp, Sm, perms, inv_perms, 
                                        M_number, beta_penalty)

            try:
                result = minimize(
                    lambda p: loss_func_wrapper(p)[0],  
                    x0,
                    method='L-BFGS-B',  
                    jac=lambda p: loss_func_wrapper(p)[1],  
                    options={'maxiter': maxiter, 'ftol': 1e-10, 'gtol': 1e-8},
                    bounds=[(-3, 3)] * total_params 
                )

                final_params = result.x
                final_loss = result.fun

                params_list = []
                for layer in range(M_number):
                    start_idx = layer * params_per_layer
                    re_part = final_params[start_idx]
                    im_part = final_params[start_idx + 1]
                    params_list.append([re_part, im_part])
                
                psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)

                psi_numpy = psi.cpu().detach().numpy()
                norm = np.vdot(psi_numpy, psi_numpy)
                
                if sp.issparse(H):
                    H_psi = H.dot(psi_numpy)
                else:
                    H_psi = H @ psi_numpy
                    
                energy = np.real(np.vdot(psi_numpy, H_psi) / norm)

                overlap1 = np.abs(np.vdot(psi_numpy, excited1_state_tensor.cpu().numpy()))**2 / norm
                overlap2 = np.abs(np.vdot(psi_numpy, excited2_state_tensor.cpu().numpy()))**2 / norm
                overlap3 = np.abs(np.vdot(psi_numpy, excited3_state_tensor.cpu().numpy()))**2 / norm
                fidelity = overlap1 + overlap2 + overlap3
                
                print(f"    Loss value: {final_loss:.8f}")
                print(f"    Energy expectation: {energy:.8f}")
                print(f"    Fidelity (sum of squares with 1st,2nd,3rd excited states): {fidelity:.8f}")

                if energy < best_energy and fidelity > 0.5:  
                    best_energy = energy
                    best_params = final_params
                    best_psi = psi_numpy
                    best_fidelity = fidelity
                    
            except Exception as e:
                print(f"    Optimization failed: {e}")
                continue
    
    return best_energy, best_params, best_psi, exact_states, best_fidelity

if __name__ == "__main__":
    L = 10
    delta = 1.0
    lambda_val = 0.1
    M_number = 4         
    num_attempts = 5
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    start_time = time.time()
    
    best_energy, best_params, best_psi, exact_states, best_fidelity = optimize_excited_state_fast(
        L=L,
        delta=delta,
        lambda_val=lambda_val,
        M_number=M_number,
        num_attempts=num_attempts,
        maxiter=5000,  
        device=device
    )
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    if best_params is not None:
        print(f"\nBethe ansatz excited state energy: {best_energy:.8f}")
        print(f"Exact first excited state energy:    {exact_states[1]['energy']:.8f}")
        print(f"Exact second excited state energy:    {exact_states[2]['energy']:.8f}")
        print(f"Exact third excited state energy:    {exact_states[3]['energy']:.8f}")
        print(f"Energy difference (with first excited state):  {abs(best_energy - exact_states[1]['energy']):.8f}")
        
        print(f"\nFidelity (sum of squares with 1st,2nd,3rd excited states): {best_fidelity:.8f}")
 
        ground_overlap = np.abs(np.vdot(best_psi, exact_states[0]['state']))
        print(f"Overlap with ground state:          {ground_overlap:.8e}")

        for i in range(1, 4):  
            overlap = np.abs(np.vdot(best_psi, exact_states[i]['state']))
            print(f"Overlap with excited state {i}:      {overlap:.8f}")
        
        print(f"\nOptimized parameters:")
        params_per_layer = 2
        for layer in range(M_number):
            start_idx = layer * params_per_layer
            re_part = best_params[start_idx]
            im_part = best_params[start_idx + 1]
            u = re_part + 1j * im_part
            print(f"  Layer {layer+1}: u = {u.real:.6f} + {u.imag:.6f}i")
        
        print(f"\nOptimization time:              {elapsed_time:.2f} seconds")
    else:
        print("Optimization failed, no valid solution found.")